In [0]:
-- Business Question 2: How does weather affect taxi demand and trip behavior?
-- Grain: Temperature band × precipitation flag × weather condition × hour
-- All widgets on the Weather Impact page read from this single SQL dataset.
WITH trip_weather AS (
  SELECT
    w.temperature_band,
    w.is_raining AS precipitation_flag,
    CASE
      WHEN w.is_raining = TRUE THEN 'Rain'
      ELSE 'No Rain'
    END AS weather_condition,
    h.hour_of_day AS hour,
    t.fare_amount,
    t.trip_distance,
    t.trip_duration_minutes,
    t.total_amount,
    t.pickup_date_key
  FROM
    nyc_mobility.gold.fact_trip AS t
      JOIN nyc_mobility.gold.dim_weather AS w
        ON t.weather_key = w.weather_key
      JOIN nyc_mobility.gold.dim_hour AS h
        ON t.pickup_hour_key = h.hour_key
),
hourly AS (
  SELECT
    temperature_band,
    precipitation_flag,
    weather_condition,
    hour,
    COUNT(*) AS trip_count,
    SUM(fare_amount) AS total_fare,
    AVG(trip_distance) AS avg_trip_distance,
    AVG(trip_duration_minutes) AS avg_trip_duration,
    AVG(trip_duration_minutes) AS avg_trip_duration_minutes,
    AVG(fare_amount) AS avg_fare,
    AVG(total_amount) AS avg_total_amount,
    COUNT(DISTINCT pickup_date_key) AS weather_hours
  FROM
    trip_weather
  GROUP BY
    temperature_band,
    precipitation_flag,
    weather_condition,
    hour
)
SELECT
  temperature_band,
  precipitation_flag,
  weather_condition,
  hour,
  trip_count,
  total_fare,
  ROUND(avg_trip_distance, 2) AS avg_trip_distance,
  ROUND(avg_trip_duration, 1) AS avg_trip_duration,
  ROUND(avg_trip_duration_minutes, 2) AS avg_trip_duration_minutes,
  ROUND(avg_fare, 2) AS avg_fare,
  ROUND(avg_total_amount, 2) AS avg_total_amount,
  ROUND(
    SUM(trip_count) OVER (PARTITION BY temperature_band, precipitation_flag, weather_condition)
      * 1.0
      / NULLIF(
        SUM(weather_hours) OVER (
            PARTITION BY temperature_band, precipitation_flag, weather_condition
          ),
        0
      ),
    2
  ) AS avg_trips_per_hour
FROM
  hourly
ORDER BY
  CASE temperature_band
    WHEN 'Below 0 C' THEN 1
    WHEN '0-10 C' THEN 2
    WHEN '10-20 C' THEN 3
    WHEN '20-30 C' THEN 4
    WHEN '30+ C' THEN 5
    ELSE 6
  END,
  weather_condition,
  hour;